# Regressione con la cover totale (cvt = cvh + cvl)

Replica il trio 03 (`delta_cover` vs `delta_skill_tas`) + 04 (vs
`delta_skill_albedo`) + 05 (vs `delta_skill_neve`), ma usando la cover
**totale** invece di cvh e cvl separate:

```
cvt = cvh + cvl
```

Nessun file dedicato per la cover totale: si costruisce sommando le due
componenti gia' allineate sugli stessi anni (vedi `_load_cover(exp, 'cvt')`
in `cover_tas_lib.py` — sostituisce trasparentemente `_load_cover(exp,'cvh')`/
`_load_cover(exp,'cvl')` ovunque, riusando le stesse funzioni di calcolo di
03/04/05 (`run_one_hybrid`, `run_one_cover_albedo`, `run_one_cover_snow`)
senza duplicare logica: basta passare `var='cvt'`.

```
X = delta_cover_totale = cvt_SENS - cvt_CTRL (anomalia, non circolare, come 01/03/04/05)
Y = delta_skill_tas / delta_skill_albedo / delta_skill_neve (genuini, vs ERA5/GLASS/ERA5)
```

Tre batch separati sotto (uno per Y), stessa maschera a soglia fissa `1e-3`
su `delta_cover` di 01/03/04/05 (la soglia era stata scelta guardando la
distribuzione di cvh/cvl separate — da verificare che sia ancora sensata per
cvt, la cui scala e' circa doppia; vedi diagnostica prima del primo batch).


In [ ]:
# rende config.py (in notebooks/) importabile anche da questa sottocartella
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025

exp_ctrl = 'a1ua'
exp_sens = 'a52o'
era_var = '2t'
snow_var = 'snd'  # 'sd' dovrebbe essere equivalente, cambiare qui se serve
variables = ['cvt']
SAVE_PATH = str(FIG_DIR)


In [ ]:
# La logica di calcolo sta in cover_tas_lib.py (stessa di 03/04/05): nessuna
# funzione nuova, si riusano run_one_hybrid/run_one_cover_albedo/
# run_one_cover_snow passando var='cvt'. _load_cover('cvt') somma cvh+cvl.
sys.path.insert(0, os.getcwd())
from cover_tas_lib import (
    run_one_hybrid, run_one_cover_albedo, run_one_cover_snow,
    debug_cover_variance, LEADS,
)


In [ ]:
# DIAGNOSTICA (eseguire prima dei batch): la soglia 1e-3 di _mask_low_variance
# era stata scelta guardando std(delta_cover) per cvh/cvl separate. cvt=cvh+cvl
# ha una scala diversa (circa doppia, se non correlate quasi il doppio della
# varianza) - verificare che la stessa soglia sia ancora sensata, non dare per
# scontato che vada bene solo perche' ha funzionato per cvh/cvl.
debug_cover_variance(exp_ctrl, exp_sens, 'cvt')


## Batch 1/3 — delta_cover_totale vs delta_skill_tas (come notebook 03)

In [ ]:
%%time
import multiprocessing as mp

jobs = [(exp_ctrl, exp_sens, var, era_var, y1, y2, SAVE_PATH) for var in variables for (y1, y2) in LEADS]

with mp.get_context('spawn').Pool(processes=2, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(run_one_hybrid, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)


## Batch 2/3 — delta_cover_totale vs delta_skill_albedo (come notebook 04)

In [ ]:
%%time
import multiprocessing as mp

jobs = [(exp_ctrl, exp_sens, var, y1, y2, SAVE_PATH) for var in variables for (y1, y2) in LEADS]

with mp.get_context('spawn').Pool(processes=2, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(run_one_cover_albedo, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)


## Batch 3/3 — delta_cover_totale vs delta_skill_neve (come notebook 05)

In [ ]:
%%time
import multiprocessing as mp

jobs = [(exp_ctrl, exp_sens, var, y1, y2, SAVE_PATH, snow_var) for var in variables for (y1, y2) in LEADS]

with mp.get_context('spawn').Pool(processes=2, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(run_one_cover_snow, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)
